# Train Point-SAM on real forest crops

Fine-tunes Point-SAM on the ISPRS Wytham **train** strip and picks the checkpoint on the **val** strip. The **test** strip is not used here: the benchmark notebook scores it once, at the end.

**Needs a CUDA GPU.** It runs top to bottom, either interactively in Jupyter on a GPU node, or unattended:
`sbatch scripts/slurm/run_notebook.sbatch notebooks/02_train.ipynb configs/experiment/isprs_lora.yaml`

Everything logged also goes to `history.csv` as it happens, so if a session drops, the last section can still draw the curves.

## 1. Setup and config

The config is the record of the run, so settings are changed there and not in this notebook.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import numpy as np
import torch
import yaml
from IPython.display import Image, display
from torch.utils.data import Subset

from treeseg.data.dataset import ForestCrops
from treeseg.engine.trainer import train
from treeseg.models.point_sam import add_point_sam_to_path, build_point_sam, prepare_for_lora
from treeseg.utils.curves import make_all

config_path = os.environ.get("TREESEG_CONFIG", "configs/experiment/smoke.yaml")
cfg = yaml.safe_load(open(config_path))
add_point_sam_to_path()
print("config:", config_path)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: this notebook will fail at training")

## 2. Data

Crops had their ground removed by the cloth filter when they were built, the same way as in the benchmark, so understory stays in. Clicks during validation come from the true labels, so these numbers measure masks, not automatic clicking.

In [ ]:
val_full = ForestCrops(os.path.join(cfg["data_dir"], "val"), cfg["num_points"], cfg["val_num_masks"],
                       cfg["min_mask_points"], seed=cfg["seed"] + 1)
val_set = Subset(val_full, range(min(cfg["val_max_crops"], len(val_full))))
print(f"val crops: {len(val_set)}")

## 3. Data augmentation

Random changes to each training crop, so the model sees different versions of the same trees and doesn't just memorise 182 crops. Validation is never augmented.

| Setting | What it simulates |
|---|---|
| `p_density` | A lower-density scan: randomly thins the crop |
| `p_occlusion` | Parts hidden from the scanner: punches box-shaped holes |
| `p_height` | Point loss by height: thins the canopy top or the stems |
| `jitter` | Scanner noise: nudges every point slightly |
| `tilt_deg` | Small tilts in the scan |
| always, when on | A random rotation about the vertical, and a 0.85-1.05 scale |

**To turn it off**, set `use_augmentation: false` in the config, or use `configs/experiment/isprs_lora_noaug.yaml`. Everything else stays the same, so comparing the two runs' test scores measures what augmentation adds. With it off, each epoch still samples different points and target trees, because the model can only take 10,000 points at a time.

In [ ]:
import matplotlib.pyplot as plt

augmentation = cfg["augmentation"] if cfg["use_augmentation"] else None
print("data augmentation:", "ON" if augmentation else "OFF", augmentation or "")
train_set = ForestCrops(os.path.join(cfg["data_dir"], "train"), cfg["num_points"], cfg["num_masks"], cfg["min_mask_points"],
                        augmentation=augmentation, resample_each_epoch=True, seed=cfg["seed"])

steps_per_epoch = len(train_set) / (cfg["batch_size"] * cfg["grad_accum"])
print(f"train crops: {len(train_set)}")
print(f"{cfg['max_steps']} steps is about {cfg['max_steps'] / steps_per_epoch:.1f} epochs")

#the same crop on three passes, from the side. With augmentation on they differ visibly
fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=110, sharey=True)
for epoch, ax in enumerate(axes):
    train_set.epoch = epoch
    item = train_set[20]
    ax.scatter(item["coords"][:, 0], item["coords"][:, 2], s=0.4, c="#52514e", linewidths=0)
    ax.set_title(f"crop 20, pass {epoch + 1}", loc="left", fontsize=10)
    ax.set_aspect("equal")
train_set.epoch = 0
plt.show()

print({key: tuple(value.shape) for key, value in item.items()})
print("share of points in each target mask:", item["gt_masks"].float().mean(dim=1).round(decimals=3).tolist())

## 4. How clicks work in training

Point-SAM doesn't find trees by itself: a click tells it "outline the thing here". During training, clicks are placed using the true labels, so the model learns "given a click on this tree, find this tree". Each step, the first click is one of these:

| Kind | Share | Where it lands |
|---|---|---|
| `stem` | 40% | Two clicks: on the trunk about 1.3 m up, and about 4 m up |
| `stem1` | 20% | The trunk click alone |
| `random` | 20% | Any point on the tree |
| `interior` | 20% | Point-SAM's own choice, the deepest point inside the tree |

After that, more clicks are added where the mask is wrong. The plot below shows where the first three kinds land on one validation tree, so it stays the same on every run. Change `DEMO_TREE` (0 to 3) to look at another, and `DEMO_CROP` for another crop.

In [ ]:
from treeseg.engine.click_demo import click_positions, predict_one_click
from treeseg.utils.view3d import plot_click_kinds, plot_mask_candidates

DEMO_CROP, DEMO_TREE = 20, 0
demo_item = val_full[DEMO_CROP]
print(f"target tree is {demo_item['gt_masks'][DEMO_TREE].float().mean():.1%} of the crop's points")
plot_click_kinds(demo_item, click_positions(demo_item, DEMO_TREE), DEMO_TREE, f"Crop {DEMO_CROP}, tree {DEMO_TREE}: where training clicks land")
plt.show()

## 5. Model

Stock Point-SAM with LoRA adapters. Only the adapters, patch embedding and decoders train.

In [ ]:
model = build_point_sam(cfg["checkpoint"], cfg["group_number"], cfg["group_size"])
prepare_for_lora(model, cfg["lora_rank"], cfg["lora_alpha"])

## 6. What one click gives, before training

The first panel is the true tree. The others are the 3 masks the stock model returns for one trunk click, the green diamond. **IoU** is how closely each matches the true tree. **Model score** is the model's own confidence, and in real use it keeps the highest-scoring mask, labelled as its pick. If the pick isn't the mask with the best IoU, a better outline existed and it didn't choose it.

In [ ]:
torch.cuda.empty_cache()
before = predict_one_click(model, demo_item, DEMO_TREE, "stem1")
print(f"model's pick: mask {before['picked'] + 1}, IoU {before['ious'][before['picked']]:.3f}.  Best available: mask {before['best'] + 1}, IoU {before['ious'][before['best']]:.3f}")
plot_mask_candidates(before, "Before training")
plt.show()
model.train();

## 7. Train

Saves `best.safetensors` whenever `val_iou_pick` improves, and stops early after 6 validations without improvement.

Every logged step prints the epoch, the step, the training loss so far this epoch, and the time left. Each epoch also prints a summary. Running through `sbatch`, they also go to the Slurm log, so you can watch them with `tail -F slurm_outs/slurm_<jobid>.out`.

In [ ]:
torch.cuda.reset_peak_memory_stats()
history_path = train(model, train_set, val_set, cfg, cfg["out_dir"])
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 8. Curves

This section only needs `history.csv`, so it also works on a partial run. If the session dropped, set `history_path` to `<out_dir>/history.csv` and run the imports in section 1 first.

In [ ]:
history_path = os.path.join(cfg["out_dir"], "history.csv")
paths = make_all(history_path, os.path.join(cfg["out_dir"], "figures"))
for name in ("training_loss", "validation_iou", "epoch_loss"):
    display(Image(paths[name]))

In [ ]:
import pandas as pd

history = pd.read_csv(history_path)
val = history[history["split"] == "val"]
best = val.loc[val[cfg["select_metric"]].idxmax()]
print(f"best {cfg['select_metric']} = {best[cfg['select_metric']]:.3f} at step {int(best['step'])} (epoch {best['epoch']:.1f})")
print(f"ran for {history['minutes'].max():.1f} minutes")
pd.read_csv(paths["per_epoch"]).round(4)

## 9. The same click, after training

The same tree and the same trunk click, using the best checkpoint. Compare it with section 6: did the masks get closer to the true tree, and does the model now pick the best one? This is one tree, so treat it as an illustration. `val_iou_pick` in the curves is the average over the whole val strip.

In [ ]:
del model
torch.cuda.empty_cache()
best_path = os.path.join(cfg["out_dir"], "best.safetensors")
trained = build_point_sam(best_path, cfg["group_number"], cfg["group_size"]).to("cuda")
after = predict_one_click(trained, demo_item, DEMO_TREE, "stem1")
print(f"before: pick IoU {before['ious'][before['picked']]:.3f}, best available {before['ious'][before['best']]:.3f}")
print(f"after:  pick IoU {after['ious'][after['picked']]:.3f}, best available {after['ious'][after['best']]:.3f}")
plot_mask_candidates(after, "After training")
plt.show()